# Liber · 검출기 미세조정: 책상 위에 홀로 세워진·누운 책 (Colab)

**문제**: 책상 위에 **홀로 세워진 두꺼운 책**(예: Embedded Recipes)을 신뢰도 0.05에서도 전혀 검출하지 못함
→ 학습 데이터의 책등은 대부분 **책장 안에서 다른 책과 붙어 있는** 모습이라, 빈 공간에 홀로 선 책·흰 배경과 대비가 낮은 책을 배운 적이 없음

**함께 고치는 문제** (테스트하며 하나씩 추가)
| # | 문제 (테스트 사진) | 원인 | 이 노트북의 대응 |
|---|---|---|---|
| 1 | 책상 위에 **홀로 세워진 두꺼운 책**을 못 찾음 (Embedded Recipes 세운 사진) | 책장 밖에 홀로 선 책·흰 배경을 배운 적 없음 | 책 없는 배경 위에 책등을 세운 합성 (두꺼운 책 · 밝은 배경 포함) |
| 2 | 책상에 **평평하게 놓인 표지 1권**을 2권으로 셈 — 책 옆 **나무 책상 띠**를 책등으로 잡음 | 책이 하나도 없는 배경 사진을 학습한 적이 거의 없음 | **책 없는 배경 사진 600장을 빈 라벨로** 학습 + 위에서 내려다본 **평평한 표지** 합성 |
| 3 | 책장 17권: 권수는 맞지만 Liber 제목에 **저자 이름이 줄줄이 붙거나**(회로이론 → `Circuit Analysis 안현식 구성철 …`) **큰 제목이 빠짐**(디지털 논리 → `entals`) | 제목 조립 규칙: 책등 폭을 꽉 채운 큰 제목을 '경계 잘림'으로 버림 · 구분 기호 없는 저자 목록을 못 거름 · 권차(상/하)를 매칭에 안 씀 | **학습 아님 → 규칙 수정** (웹앱 Liber·맥/Colab 노트북 Cell 9-1): 경계 잘림은 가로 줄만 · 세 글자 이름 3개 이상 줄은 저자 · 권차 붙인 질의 |
| 4 | 서점 진열대 27권: 권수는 맞지만 표지 제목이 어긋남 (`트렌드 코리아` 2027 빠짐 · `교과서 부동산` 어순 뒤집힘 · 광고 문구가 붙음 · `Atomic Habits`의 한글 제목 못 찾음) | 숫자만 있는 줄을 버림 · 표지 제목 기준이 너그러움 · 어순 교정 과신 · 카탈로그 매칭을 제목 줄에만 | **규칙 수정** (웹앱·노트북 Block 9): 큰 숫자 줄 포함 · 표지는 75%·최대 3줄 · 어순은 카탈로그 점수 +10 때만 · 모든 줄로 카탈로그 확정(88점) |
| 5 | 위에서 찍은 **옆으로 눕힌 표지**(회로이론): 책 전체가 아니라 **오른쪽 초록 제목 띠만** 상자로 잡음 | 세로 띠 + 세로 글씨 = 책등이라고 강하게 배움 · 위에서 본 표지 전체를 상자로 잡는 예가 적음 | 합성에서 표지 50%(←35%) · 그중 평평한 표지 65%(←50%) · **옆으로 눕힌 표지(90°) 25%** — 라벨은 항상 표지 전체 |

**해결**: 실제 책등 크롭(Roboflow 정답 상자) · Kaggle 표지를 **책 없는 책상·사무실 사진** 위에 1~3권씩 세우고·눕히고·쌓아 붙인 **합성 사진**을 만들어,
본학습 검출기(`det_obb_best.pt`)에서 **이어서 학습**한다. 기존 학습 데이터도 함께 써서 원래 잘하던 것(빽빽한 책장)을 잊지 않게 한다.

| 단계 | 내용 | 예상 시간 (A100) |
|---|---|---|
| 준비 | Drive의 데이터셋·가중치 복원 · 책 없는 장면 2,500장 · Kaggle 표지 | 10~15분 |
| 합성 | 학습용 4,000장 + 확인용 300장 (정확한 회전 상자 라벨) | 5~10분 |
| 미세조정 | 최대 30에폭 · 10에폭 동안 개선 없으면 멈춤 · 낮은 학습률 | 1~1.5시간 |
| 평가 | 미세조정 전 ↔ 후: 권수 구간별 · 합성 확인용 · **테스트 사진(samples/) 그림 비교** | 5~10분 |
| 웹앱용 | 신뢰도 기준 다시 고르기 · `liber_web_models_ft.zip` 자동 다운로드 | 2~3분 |

### 필요한 것 (Drive `내 드라이브/liber/` — 본학습 노트북이 이미 만들어 둔 것)
`books_obb_built.zip` · `det_obb_best.pt` · `archive.zip`(Kaggle 표지, 없으면 책등만) · `samples/`(테스트 사진, 학습에 쓰지 않음)

런타임 **A100 GPU** (고용량 RAM 권장) → **모두 실행**

In [ ]:
# Cell 0 · 패키지
!pip -q install ultralytics==8.4.183 onnx onnxruntime onnxslim pillow-heif
!apt-get -qq install -y fonts-nanum > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 1 · 준비: Drive · 경로 · 데이터셋·가중치 복원
import json
import os
import random
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from google.colab import drive, files
from matplotlib import font_manager
from PIL import Image, ImageOps
from ultralytics import YOLO

drive.mount("/content/drive")
for f in Path("/usr/share/fonts/truetype/nanum").glob("*.ttf"):
    font_manager.fontManager.addfont(str(f))
plt.rcParams["font.family"] = "NanumGothic"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
TIMES = {}
DEVICE = 0

DRIVE = Path("/content/drive/MyDrive/liber")
WORK = Path("/content/work_ft")
DS = WORK / "books_obb"
WORK.mkdir(parents=True, exist_ok=True)
BASE_PT = DRIVE / "det_obb_best.pt"          # 본학습 결과 (미세조정 시작점)
FT_RUN = DRIVE / "runs" / "det_obb_ft"
FT_OUT = DRIVE / "det_obb_best_ft.pt"
IMGSZ = 960
for need in (DRIVE / "books_obb_built.zip", BASE_PT):
    assert need.exists(), f"{need} 없음 → 본학습 노트북(colab_liber_full.ipynb)을 먼저 실행하세요"

t0 = time.time()
if not DS.exists():
    subprocess.run(["unzip", "-q", "-o", str(DRIVE / "books_obb_built.zip"), "-d", str(WORK)], check=True)
cfg = yaml.safe_load(open(DS / "data.yaml"))
cfg["path"] = str(DS)
yaml.safe_dump(cfg, open(DS / "data.yaml", "w"), sort_keys=False)
TIMES["데이터셋 복원"] = (time.time() - t0) / 60
print({s: len(list((DS / "images" / s).iterdir())) for s in ("train", "val")})

In [ ]:
# Cell 2 · 공통 함수 (본학습 노트북과 같은 라벨 형식·회전 크롭·평가)
def write_obb_label(path, obbs):
    path.write_text("\n".join("0 " + " ".join(f"{v:.6f}" for v in o) for o in obbs) + "\n")


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in Path(lbl_path).read_text().splitlines() if len(l.split()) >= 9]


def rect_frame(poly):
    box = cv2.boxPoints(cv2.minAreaRect(np.asarray(poly, np.float32)))
    for i in range(4):
        for step in (1, -1):
            p0, p1, p3 = box[i], box[(i + step) % 4], box[(i - step) % 4]
            u, v = p1 - p0, p3 - p0
            if np.linalg.norm(u) >= np.linalg.norm(v) and v[0] * u[1] - v[1] * u[0] > 0:
                return p0, u, v
    p0, p1, p3 = box[0], box[1], box[3]
    return p0, p1 - p0, p3 - p0


def crop_rotated(img, poly):
    """회전 상자 → 긴 변이 세로가 되게 바로 세운 크롭"""
    p0, u, v = rect_frame(poly)
    L, S = max(int(round(np.linalg.norm(u))), 1), max(int(round(np.linalg.norm(v))), 1)
    M = cv2.getAffineTransform(np.float32([p0, p0 + u, p0 + v]), np.float32([[0, 0], [0, L], [S, 0]]))
    return cv2.warpAffine(img, M, (S, L), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def limit_side(img, max_side):
    s = max_side / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def poly_iou(a, b):
    inter, _ = cv2.intersectConvexConvex(a, b)
    union = cv2.contourArea(a) + cv2.contourArea(b) - inter
    return inter / union if union > 0 else 0.0


def bucket(n):
    return f"{n}권" if n <= 3 else ("4~9권" if n <= 9 else "10권+")

In [ ]:
# Cell 3 · [재료 ①] 책 없는 장면 (Open Images: 책상·사무실·탁자 등, 책·책장·선반이 찍힌 사진 제외) → 합성 배경
OI_BASE = "https://storage.googleapis.com/openimages"
BG_CLASSES = ["Table", "Desk", "Kitchen & dining room table", "Countertop", "Computer monitor", "Laptop", "Television",
              "Cabinetry", "Chest of drawers", "Whiteboard", "Picture frame", "Office building", "Window", "Wall clock", "Couch"]
EXCLUDE = ["Book", "Bookcase", "Shelf"]
N_BG = 2500
t0 = time.time()
csv_dir = WORK / "oi_csv"
csv_dir.mkdir(exist_ok=True)
urls = {"classes": f"{OI_BASE}/v7/oidv7-class-descriptions-boxable.csv",
        "validation": f"{OI_BASE}/v5/validation-annotations-bbox.csv", "test": f"{OI_BASE}/v5/test-annotations-bbox.csv"}
for u in urls.values():
    if not (csv_dir / Path(u).name).exists():
        subprocess.run(["wget", "-q", "-O", str(csv_dir / Path(u).name), u], check=True)
names = pd.read_csv(csv_dir / Path(urls["classes"]).name)
mid = dict(zip(names.DisplayName, names.LabelName))
ann = pd.concat([pd.read_csv(csv_dir / Path(urls[s]).name, usecols=["ImageID", "LabelName"]).assign(split=s) for s in ("validation", "test")])
bad = set(ann.ImageID[ann.LabelName.isin([mid[n] for n in EXCLUDE if n in mid])])
good = ann[ann.LabelName.isin([mid[n] for n in BG_CLASSES if n in mid]) & ~ann.ImageID.isin(bad)]
picks = good.drop_duplicates("ImageID").sample(n=min(N_BG, good.ImageID.nunique()), random_state=SEED)
bg_dir = WORK / "bg"
bg_dir.mkdir(exist_ok=True)


def fetch_bg(row):
    out = bg_dir / f"{row.ImageID}.jpg"
    if not out.exists():
        try:
            r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{row.split}/{row.ImageID}.jpg", timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            return None
        img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            return None
        cv2.imwrite(str(out), limit_side(img, 1280), [cv2.IMWRITE_JPEG_QUALITY, 92])
    return out


with ThreadPoolExecutor(32) as ex:
    BG = [p for p in ex.map(fetch_bg, picks.itertuples(index=False)) if p is not None]
TIMES["배경 다운로드"] = (time.time() - t0) / 60
print(f"배경 {len(BG):,}장")

In [ ]:
# Cell 4 · [재료 ②] 책 이미지 모음: 실제 책등 크롭(Roboflow 정답 상자, 학습용 사진에서만) + Kaggle 표지
N_SPINE, N_COVER = 3000, 3000
t0 = time.time()
rng = random.Random(SEED)
spines = []
train_imgs = [p for p in sorted((DS / "images" / "train").iterdir()) if p.stem.startswith("rf_") and "_rep" not in p.stem]
rng.shuffle(train_imgs)
for p in train_imgs:
    if len(spines) >= N_SPINE:
        break
    img = cv2.imread(str(p))
    if img is None:
        continue
    polys = read_obb(DS / "labels" / "train" / f"{p.stem}.txt", img.shape[1], img.shape[0])
    for poly in rng.sample(polys, min(2, len(polys))):
        c = crop_rotated(img, poly)
        if c.shape[0] >= 120 and c.shape[0] >= 2.5 * c.shape[1]:   # 충분히 크고 책등처럼 긴 것만
            spines.append(limit_side(c, 900))

covers = []
cov_zip = next((z for z in (DRIVE / "archive.zip", DRIVE / "book_covers.zip") if z.exists()), None)
if cov_zip:
    cdir = WORK / "covers"
    if not cdir.exists():
        subprocess.run(["unzip", "-q", "-o", str(cov_zip), "book-covers/*", "-d", str(cdir)], check=True)
    cps = sorted(cdir.rglob("*.jpg"))
    rng.shuffle(cps)
    for p in cps[:N_COVER]:
        c = cv2.imread(str(p))
        if c is not None and min(c.shape[:2]) >= 100:
            covers.append(c)
TIMES["책 이미지 모음"] = (time.time() - t0) / 60
print(f"책등 {len(spines):,} · 표지 {len(covers):,}")

In [ ]:
# Cell 5 · [합성] 배경 위에 책 1~3권: 세움(두꺼운 책 포함) · 누움 · 쌓음 · 표지 · 그림자 · 밝은 배경(대비 낮음)
#   라벨 = 붙인 책 네 꼭짓점의 회전 사각형 (정확). 사진 밖으로 15% 넘게 나가거나 다른 책을 15% 넘게 가리는 배치는 버리고 다시
N_SYN, N_SYN_VAL = 4000, 300
N_NEG, N_NEG_VAL = 600, 100   # 책이 하나도 없는 배경 사진 (빈 라벨) → '빈 배경은 책이 아님'을 학습 (문제 2)


def rotate_obj(obj, ang):
    h, w = obj.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    c, s = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * s + w * c) + 2, int(h * c + w * s) + 2
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    rot = cv2.warpAffine(obj, M, (nw, nh), flags=cv2.INTER_LINEAR)
    mask = cv2.warpAffine(np.full((h, w), 255, np.uint8), M, (nw, nh), flags=cv2.INTER_LINEAR)
    corners = cv2.transform(np.float32([[[0, 0], [w, 0], [w, h], [0, h]]]), M)[0]
    return rot, mask, corners


def paste(canvas, rot, mask, x0, y0, r):
    H, W = canvas.shape[:2]
    nh, nw = mask.shape
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + nw), min(H, y0 + nh)
    if xb <= xa or yb <= ya:
        return
    m = mask[ya - y0:yb - y0, xa - x0:xb - x0].astype(np.float32) / 255.0
    o = rot[ya - y0:yb - y0, xa - x0:xb - x0].astype(np.float32) * r.uniform(0.7, 1.15)   # 책 밝기 변화
    sh = cv2.GaussianBlur(m, (0, 0), max(2.0, 0.04 * min(nw, nh)))                         # 부드러운 그림자 (오른쪽 아래로)
    dx, dy = int(0.03 * nw) + 1, int(0.02 * nh) + 1
    sh = np.pad(sh, ((dy, 0), (dx, 0)))[:m.shape[0], :m.shape[1]]
    region = canvas[ya:yb, xa:xb].astype(np.float32) * (1 - 0.35 * sh[..., None])
    canvas[ya:yb, xa:xb] = np.clip(region * (1 - m[..., None]) + o * m[..., None], 0, 255).astype(np.uint8)


def make_one(r):
    bg = cv2.imread(str(r.choice(BG)))
    H, W = bg.shape[:2]
    if r.random() < 0.3:   # 흰 벽·밝은 모니터 같은 밝은 배경 → 흰 책과 대비가 낮은 장면
        a = r.uniform(0.3, 0.6)
        bg = (bg.astype(np.float32) * (1 - a) + 255 * a).astype(np.uint8)
    canvas = bg.copy()
    frame = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
    placed = []
    desk = r.uniform(0.6, 0.95) * H              # 세운 책이 놓이는 책상 높이
    stack_y = None
    for _ in range(r.choice([1, 1, 2, 2, 3])):
        kind = "cover" if covers and r.random() < 0.5 else "spine"
        if kind == "cover":   # 세운 표지 / 책상에 평평하게 놓인 표지(위에서 내려다본 모습: 크게) — 문제 2·5
            pose = "flat" if r.random() < 0.65 else "stand"
        else:
            pose = r.choices(["stand", "lie", "any"], [0.55, 0.35, 0.10])[0]
        for _try in range(10):
            obj = (r.choice(covers) if kind == "cover" else r.choice(spines)).copy()
            if kind == "spine" and r.random() < 0.5:   # 두꺼운 책: 책등 폭을 넓힘
                obj = cv2.resize(obj, (int(obj.shape[1] * r.uniform(1.3, 2.8)), obj.shape[0]))
            if pose == "stand":
                target, ang = r.uniform(0.3, 0.85) * H, r.uniform(-8, 8)
            elif pose == "lie":
                target, ang = r.uniform(0.3, 0.8) * W, 90 + r.uniform(-6, 6)
            elif pose == "flat":
                u = r.random()   # 정방향 근처 50% · 옆으로 눕힘(90°) 25% · 아무 각도 25% — 표지 전체가 한 권 (문제 5)
                ang = r.uniform(-25, 25) if u < 0.5 else (r.choice([90, -90]) + r.uniform(-10, 10)) if u < 0.75 else r.uniform(0, 360)
                target = r.uniform(0.45, 0.9) * min(H, W)
            else:
                target, ang = r.uniform(0.25, 0.7) * min(H, W), r.uniform(0, 180)
            s = target / obj.shape[0]
            obj = cv2.resize(obj, (max(8, int(obj.shape[1] * s)), max(8, int(obj.shape[0] * s))), interpolation=cv2.INTER_AREA)
            rot, mask, corners = rotate_obj(obj, ang)
            nh, nw = mask.shape
            if pose == "stand":
                cx, cy = r.uniform(0.1, 0.9) * W, desk - nh / 2
            elif pose == "flat":
                cx, cy = r.uniform(0.3, 0.7) * W, r.uniform(0.3, 0.7) * H
            elif pose == "lie" and stack_y is not None and r.random() < 0.6:
                cx, cy = stack_cx + r.uniform(-0.05, 0.05) * W, stack_y - nh / 2   # 앞서 누운 책 위에 쌓음
            else:
                cx, cy = r.uniform(0.15, 0.85) * W, r.uniform(0.45, 0.92) * H
            x0, y0 = int(cx - nw / 2), int(cy - nh / 2)
            poly = (corners + np.float32([x0, y0])).astype(np.float32)
            area = cv2.contourArea(poly)
            vis, _ = cv2.intersectConvexConvex(poly, frame)
            if area <= 0 or vis / area < 0.85:
                continue
            if any(cv2.intersectConvexConvex(poly, q)[0] / min(area, cv2.contourArea(q)) > 0.15 for q in placed):
                continue
            paste(canvas, rot, mask, x0, y0, r)
            placed.append(poly)
            if pose == "lie":
                stack_y, stack_cx = poly[:, 1].min(), cx
            break
    if not placed:
        return None
    if r.random() < 0.3:
        canvas = cv2.GaussianBlur(canvas, (3, 3), 0)
    canvas = np.clip(canvas.astype(np.float32) * r.uniform(0.85, 1.1) + r.uniform(-15, 15), 0, 255).astype(np.uint8)
    return canvas, [(p / np.float32([W, H])).clip(0, 1).reshape(-1).tolist() for p in placed]


t0 = time.time()
SYN_VAL = WORK / "syn_val"
for d in (SYN_VAL / "images", SYN_VAL / "labels"):
    d.mkdir(parents=True, exist_ok=True)
n_ok = Counter()
for i in range(N_SYN + N_SYN_VAL):
    out = make_one(random.Random(SEED * 1000 + i))
    if out is None:
        continue
    img, obbs = out
    if i < N_SYN:   # 학습 데이터에 바로 추가
        name = f"syn2__{i:05d}"
        cv2.imwrite(str(DS / "images" / "train" / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, random.Random(i).randint(75, 95)])
        write_obb_label(DS / "labels" / "train" / f"{name}.txt", obbs)
        n_ok["train"] += 1
    else:           # 합성 확인용 (학습에 안 씀)
        name = f"synval__{i:05d}"
        cv2.imwrite(str(SYN_VAL / "images" / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        write_obb_label(SYN_VAL / "labels" / f"{name}.txt", obbs)
        n_ok["val"] += 1
def make_negative(r):
    """책이 없는 배경을 무작위로 잘라 크기·밝기를 바꾼 사진 (나무결·벽·모니터 같은 빈 영역)"""
    bg = cv2.imread(str(r.choice(BG)))
    H, W = bg.shape[:2]
    s = r.uniform(0.5, 1.0)
    ch, cw = int(H * s), int(W * r.uniform(0.5, 1.0))
    y, x = r.randint(0, H - ch), r.randint(0, W - cw)
    img = bg[y:y + ch, x:x + cw]
    if r.random() < 0.5:
        img = cv2.flip(img, 1)
    if r.random() < 0.3:
        a = r.uniform(0.3, 0.6)
        img = (img.astype(np.float32) * (1 - a) + 255 * a).astype(np.uint8)
    return np.clip(img.astype(np.float32) * r.uniform(0.85, 1.1) + r.uniform(-15, 15), 0, 255).astype(np.uint8)


NEG_VAL = WORK / "neg_val"
for d in (NEG_VAL / "images", NEG_VAL / "labels"):
    d.mkdir(parents=True, exist_ok=True)
for i in range(N_NEG + N_NEG_VAL):
    img = make_negative(random.Random(SEED * 7000 + i))
    if i < N_NEG:
        cv2.imwrite(str(DS / "images" / "train" / f"neg2__{i:05d}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        (DS / "labels" / "train" / f"neg2__{i:05d}.txt").write_text("")   # 빈 라벨 = 배경 사진
        n_ok["neg"] += 1
    else:
        cv2.imwrite(str(NEG_VAL / "images" / f"negval__{i:05d}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        (NEG_VAL / "labels" / f"negval__{i:05d}.txt").write_text("")
        n_ok["neg_val"] += 1

for cache in DS.rglob("*.cache"):   # 학습 목록이 바뀌었으니 라벨 캐시를 지움
    cache.unlink()
TIMES["합성"] = (time.time() - t0) / 60
print(f"합성 학습용 {n_ok['train']:,} · 확인용 {n_ok['val']:,} · 배경만 학습용 {n_ok['neg']:,} · 확인용 {n_ok['neg_val']:,} · {TIMES['합성']:.1f}분")

In [ ]:
# Cell 6 · ✅ 합성 사진 확인 (8장)
ps = sorted((DS / "images" / "train").glob("syn2__*.jpg"))
random.Random(1).shuffle(ps)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p in zip(axes.flat, ps[:8]):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for q in read_obb(DS / "labels" / "train" / f"{p.stem}.txt", w, h):
        cv2.polylines(img, [q.astype(np.int32)], True, (255, 0, 0), max(2, w // 300))
    ax.imshow(img)
    ax.set_title(p.stem, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 · [미세조정] 본학습 검출기에서 이어서: 기존 학습 데이터 + 합성 4,000장 · 낮은 학습률 · 최대 30에폭
#   세션이 끊기면 이 셀만 다시 실행 → last.pt에서 이어서 (단, Cell 1~5를 먼저 다시 실행해 데이터를 같은 자리에)
t0 = time.time()
last = FT_RUN / "weights" / "last.pt"
if last.exists():
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:
        print(f"이어서 할 학습 없음: {e}")
else:
    YOLO(str(BASE_PT)).train(
        data=str(DS / "data.yaml"), project=str(FT_RUN.parent), name=FT_RUN.name, exist_ok=True,
        epochs=30, patience=10, imgsz=IMGSZ, batch=32, workers=8, device=DEVICE, seed=SEED, cache="disk",
        optimizer="SGD", lr0=0.003, lrf=0.1, warmup_epochs=1,      # 이미 학습된 가중치라 처음보다 작은 학습률
        degrees=15.0, fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=5, save_period=-1,
    )
shutil.copy2(FT_RUN / "weights" / "best.pt", FT_OUT)
TIMES["미세조정"] = (time.time() - t0) / 60
print(f"미세조정 가중치 → {FT_OUT} · {TIMES['미세조정'] / 60:.2f}시간")
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 8 · ✅ 미세조정 전 ↔ 후: 권수 구간별(원래 검증 사진) + 합성 확인용 — 원래 잘하던 것을 잊지 않았는지 · 새 장면을 배웠는지
CONF = 0.25
models = {"전 (본학습)": YOLO(str(BASE_PT)), "후 (미세조정)": YOLO(str(FT_OUT))}


def eval_dir(model, img_dir, lbl_dir, group=None):
    rows = []
    for p in sorted(Path(img_dir).glob("*.jpg")):
        img = cv2.imread(str(p))
        gt = read_obb(Path(lbl_dir) / f"{p.stem}.txt", img.shape[1], img.shape[0])
        r = model.predict(img, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]
        pred = [q.astype(np.float32) for q in r.obb.xyxyxyxy.cpu().numpy()] if r.obb is not None else []
        used, tp = set(), 0
        for g in gt:
            cand = [(poly_iou(g, q), j) for j, q in enumerate(pred) if j not in used]
            if cand and max(cand)[0] >= 0.5:
                used.add(max(cand)[1])
                tp += 1
        rows.append({"구간": group or bucket(len(gt)), "실제": len(gt), "검출": len(pred), "맞음": tp})
    return pd.DataFrame(rows)


def summarize(ev):   # '책 없는 배경'은 권수 정확 = 상자를 하나도 안 낸 비율, 잘못 잡은 상자 = 사진당 평균
    return ev.groupby("구간").apply(lambda g: pd.Series({
        "사진 수": len(g), "권수 정확(%)": round((g["실제"] == g["검출"]).mean() * 100, 1),
        "잘못 잡은 상자(사진당)": round((g["검출"] - g["맞음"]).mean(), 2),
        "재현율(%)": round(g["맞음"].sum() / max(g["실제"].sum(), 1) * 100, 1),
        "정밀도(%)": round(g["맞음"].sum() / max(g["검출"].sum(), 1) * 100, 1)}))


t0 = time.time()
tables = {}
for k, m in models.items():
    ev = pd.concat([eval_dir(m, DS / "images" / "val", DS / "labels" / "val"),
                    eval_dir(m, SYN_VAL / "images", SYN_VAL / "labels", group="합성: 책상 위 1~3권"),
                    eval_dir(m, NEG_VAL / "images", NEG_VAL / "labels", group="책 없는 배경 (0권이 정답)")])
    tables[k] = summarize(ev)
order = ["1권", "2권", "3권", "4~9권", "10권+", "합성: 책상 위 1~3권", "책 없는 배경 (0권이 정답)"]
cmp = pd.concat(tables, axis=1).reindex([o for o in order if o in tables["전 (본학습)"].index])
display(cmp)
cmp.to_csv(DRIVE / "eval_finetune.csv", encoding="utf-8-sig")
TIMES["전후 평가"] = (time.time() - t0) / 60

In [ ]:
# Cell 9 · ✅ 테스트 사진(samples/) 전 ↔ 후 그림: 사진마다 [미세조정 전 | 미세조정 후], 상자 옆 숫자 = 신뢰도 (주황 = 0.45 미만)
from pillow_heif import register_heif_opener
register_heif_opener()
SAMPLES = WORK / "samples"
SAMPLES.mkdir(exist_ok=True)
for p in sorted((DRIVE / "samples").glob("*")) if (DRIVE / "samples").exists() else []:
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".heic", ".heif"} and not (SAMPLES / f"{p.stem}.jpg").exists():
        ImageOps.exif_transpose(Image.open(p)).convert("RGB").save(SAMPLES / f"{p.stem}.jpg", quality=95)
tests = sorted(SAMPLES.glob("*.jpg"))
SHOW_CONF = 0.1
fig, axes = plt.subplots(len(tests), 2, figsize=(14, 6 * len(tests)), squeeze=False)
for row, p in zip(axes, tests):
    img = cv2.imread(str(p))
    for ax, (k, m) in zip(row, models.items()):
        r = m.predict(img, imgsz=IMGSZ, conf=SHOW_CONF, device=DEVICE, verbose=False)[0]
        vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t = max(3, img.shape[1] // 250)
        n_hi = 0
        for q, c in zip(r.obb.xyxyxyxy.cpu().numpy(), r.obb.conf.cpu().numpy()):
            col = (255, 0, 0) if c >= 0.45 else (255, 165, 0)
            n_hi += c >= 0.45
            cv2.polylines(vis, [q.astype(np.int32)], True, col, t)
            cv2.putText(vis, f"{c:.2f}", tuple(q[0].astype(int)), cv2.FONT_HERSHEY_SIMPLEX, t * 0.5, col, t)
        ax.imshow(vis)
        ax.set_title(f"[{p.name}] {k} · 0.45 이상 {n_hi}개", fontsize=12)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 · 웹앱용: 신뢰도 기준 다시 고르기(권수 구간 + 합성 평균) → e_unified.onnx · e_unified.json → liber_web_models_ft.zip 자동 다운로드
import onnxruntime as ort

ft = models["후 (미세조정)"]
CONF_GRID = [round(x, 2) for x in np.arange(0.15, 0.61, 0.05)]
recs = []
for img_dir, lbl_dir, grp in [(DS / "images" / "val", DS / "labels" / "val", None), (SYN_VAL / "images", SYN_VAL / "labels", "합성"),
                              (NEG_VAL / "images", NEG_VAL / "labels", "배경")]:
    for p in sorted(Path(img_dir).glob("*.jpg")):
        img = cv2.imread(str(p))
        n_gt = len(read_obb(Path(lbl_dir) / f"{p.stem}.txt", img.shape[1], img.shape[0]))
        r = ft.predict(img, imgsz=IMGSZ, conf=min(CONF_GRID), device=DEVICE, verbose=False)[0]
        recs.append((grp or bucket(n_gt), n_gt, r.obb.conf.cpu().numpy() if r.obb is not None else np.array([])))
sweep = pd.DataFrame([{"conf": c, "구간": b, "정확": int((cf >= c).sum() == n)} for c in CONF_GRID for b, n, cf in recs])
table = sweep.pivot_table(index="conf", columns="구간", values="정확", aggfunc="mean").mul(100).round(1)
table["구간 평균"] = table.mean(axis=1).round(1)
WEB_CONF = float(table["구간 평균"].idxmax())
display(table)
print(f"웹앱 신뢰도 기준 → {WEB_CONF}")

WEB = WORK / "liber_web_models_ft"
if WEB.exists():
    shutil.rmtree(WEB)
WEB.mkdir()
onnx_path = YOLO(str(FT_OUT), task="obb").export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)
shutil.copy2(onnx_path, WEB / "e_unified.onnx")
(WEB / "e_unified.json").write_text(json.dumps({"imgsz": IMGSZ, "conf": WEB_CONF, "zoom": 0}))
s = ort.InferenceSession(str(WEB / "e_unified.onnx"), providers=["CPUExecutionProvider"])
o = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, IMGSZ, IMGSZ), np.float32)})[0]
assert o.shape[1] == 6, o.shape
web_zip = shutil.make_archive(str(WORK / "liber_web_models_ft"), "zip", WEB)
shutil.copy2(web_zip, DRIVE / "liber_web_models_ft.zip")
print(f"웹앱용 → {DRIVE / 'liber_web_models_ft.zip'} ({Path(web_zip).stat().st_size / 1e6:.1f}MB) · 출력 {o.shape}")
display(pd.Series(TIMES, name="분").round(1).to_frame())
files.download(web_zip)

## 웹앱에 적용
`liber_web_models_ft.zip`을 받은 뒤 맥에서:
```bash
cd ~/Downloads/shelf_ocr_test
unzip -o ~/Downloads/liber_web_models_ft.zip e_unified.onnx e_unified.json -d web/
```
→ 버전 올리기 · 커밋 · 푸시. 맥 노트북에도 쓰려면 Drive의 `det_obb_best_ft.pt`를 `liber_deprehensus/weights/det_obb_best.pt`로.